# Ancient Chinese NER — GuwenBERT Training
Train 2 models trên unified dataset (AnChineseNERE + CHisIEC + BEEVETA + C-CLUE):
- **Model A**: GuwenBERT + Linear + CRF
- **Model B**: GuwenBERT + Linear (softmax, no CRF)

Sau train: error analysis cho cả 2 models.

## Cell 1 — Setup

In [1]:
import os, subprocess, sys

os.chdir('/kaggle/working')
if not os.path.exists('chisiec-ner'):
    subprocess.run(['git', 'clone', 'https://github.com/lethanhconghcmus-alt/chisiec-ner.git'], check=True)
os.chdir('/kaggle/working/chisiec-ner')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'pytorch-crf', 'seqeval', 'omegaconf', 'tqdm'], check=True)
print('✅ Setup done', flush=True)

Cloning into 'chisiec-ner'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
✅ Setup done


## Cell 2 — Load Unified Data

In [2]:
import os, shutil
from collections import Counter

# Copy unified data from Kaggle dataset
DATA_SRC = '/kaggle/input/datasets/thnhcngl/ner-data-a/data/unified'
os.makedirs('data/ancient', exist_ok=True)

for split in ['train', 'dev', 'test']:
    shutil.copy(f'{DATA_SRC}/{split}.txt', f'data/ancient/{split}.txt')

def read_conll(path):
    sents, cur = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if cur: sents.append(cur)
                cur = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    cur.append((parts[0], parts[-1]))
    if cur: sents.append(cur)
    return sents

def build_label_map(data):
    tags = sorted(set(tag for s in data for _, tag in s))
    label2id = {t: i for i, t in enumerate(tags)}
    id2label = {i: t for t, i in label2id.items()}
    return label2id, id2label

train_data = read_conll('data/ancient/train.txt')
dev_data   = read_conll('data/ancient/dev.txt')
test_data  = read_conll('data/ancient/test.txt')
label2id, id2label = build_label_map(train_data)

print(f'Train: {len(train_data):,} sentences')
print(f'Dev:   {len(dev_data):,} sentences')
print(f'Test:  {len(test_data):,} sentences')
print(f'Labels ({len(label2id)}): {sorted(label2id.keys())}')

# Entity distribution
for split_name, data in [('Train', train_data), ('Dev', dev_data), ('Test', test_data)]:
    ec = Counter(tag[2:] for s in data for _, tag in s if tag.startswith('B-'))
    print(f'{split_name} entity dist: {dict(sorted(ec.items()))}')


Train: 11,386 sentences
Dev:   1,697 sentences
Test:  3,076 sentences
Labels (11): ['B-DTM', 'B-LOC', 'B-ORG', 'B-PER', 'B-TITLE', 'I-DTM', 'I-LOC', 'I-ORG', 'I-PER', 'I-TITLE', 'O']
Train entity dist: {'DTM': 1574, 'LOC': 14638, 'ORG': 2090, 'PER': 30314, 'TITLE': 7086}
Dev entity dist: {'DTM': 237, 'LOC': 1864, 'ORG': 15, 'PER': 3898, 'TITLE': 1311}
Test entity dist: {'DTM': 176, 'LOC': 5029, 'ORG': 2047, 'PER': 13865, 'TITLE': 2807}


## Cell 3 — Model Definitions

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torchcrf import CRF

BACKBONE = 'ethanyt/guwenbert-base'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}', flush=True)

# ── Model A: GuwenBERT + CRF ───────────────────────────────────────
class GuwenBertCRF(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(backbone)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf  = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None and getattr(self.bert.config, 'type_vocab_size', 1) > 1:
            kwargs['token_type_ids'] = token_type_ids
        seq    = self.drop(self.bert(**kwargs).last_hidden_state)
        logits = self.fc(seq)
        mask   = attention_mask.bool()
        mask[:, 0] = True
        if labels is not None:
            crf_labels = labels.clone()
            crf_labels[crf_labels == -100] = 0
            loss = -self.crf(logits, crf_labels, mask=mask, reduction='mean')
            return loss, logits
        return self.crf.decode(logits, mask=mask)

# ── Model B: GuwenBERT + Linear (no CRF) ──────────────────────────
class GuwenBertLinear(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(backbone)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None and getattr(self.bert.config, 'type_vocab_size', 1) > 1:
            kwargs['token_type_ids'] = token_type_ids
        seq    = self.drop(self.bert(**kwargs).last_hidden_state)
        logits = self.fc(seq)
        if labels is not None:
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            return loss, logits
        return logits.argmax(dim=-1)

num_labels = len(label2id)
print(f'num_labels: {num_labels}')
print('✅ Models defined', flush=True)

Device: cuda
num_labels: 11
✅ Models defined


## Cell 4 — Dataset & DataLoader

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class AncientNERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.data      = data
        self.tokenizer = tokenizer
        self.label2id  = label2id
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = zip(*self.data[idx])
        tokens, tags = list(tokens), list(tags)

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids      = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        token_type_ids = encoding.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.squeeze(0)

        word_ids = encoding.word_ids(batch_index=0)
        labels   = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev_word_id:
                labels.append(self.label2id.get(tags[wid], self.label2id['O']))
            else:
                labels.append(-100)
            prev_word_id = wid

        item = {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'labels':         torch.tensor(labels, dtype=torch.long),
        }
        if token_type_ids is not None:
            item['token_type_ids'] = token_type_ids
        return item

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
MAX_LEN, BS = 128, 16

train_loader = DataLoader(AncientNERDataset(train_data, tokenizer, label2id, MAX_LEN),
                          batch_size=BS, shuffle=True,  num_workers=2)
dev_loader   = DataLoader(AncientNERDataset(dev_data,   tokenizer, label2id, MAX_LEN),
                          batch_size=BS, shuffle=False, num_workers=2)
test_loader  = DataLoader(AncientNERDataset(test_data,  tokenizer, label2id, MAX_LEN),
                          batch_size=BS, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Dev: {len(dev_loader)} | Test: {len(test_loader)}')
print('✅ DataLoaders ready', flush=True)

config.json:   0%|          | 0.00/519 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Train batches: 712 | Dev: 107 | Test: 193
✅ DataLoaders ready


## Cell 5 — Train & Evaluate Functions

In [5]:
import time
from torch.amp import GradScaler, autocast
from transformers import get_linear_schedule_with_warmup
from seqeval.metrics import f1_score, classification_report

def evaluate(model, loader, use_crf=True):
    model.eval()
    all_preds, all_golds = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            labels = batch['labels']
            preds  = model(input_ids, attention_mask, token_type_ids)

            for pred_seq, label_seq, mask in zip(preds, labels, attention_mask):
                pred_list, gold_list = [], []
                if use_crf:
                    pred_iter = pred_seq
                else:
                    pred_iter = pred_seq.tolist()
                for p, l, m in zip(pred_iter, label_seq.tolist(), mask.tolist()):
                    if m == 0: break
                    if l == -100: continue
                    pred_list.append(id2label[p])
                    gold_list.append(id2label[l])
                all_preds.append(pred_list)
                all_golds.append(gold_list)
    return f1_score(all_golds, all_preds), all_preds, all_golds

def train_model(model, model_name, epochs=20, bert_lr=1e-5, head_lr=1e-3, patience=5):
    model.to(DEVICE)
    use_crf = hasattr(model, 'crf')
    
    optimizer = torch.optim.AdamW([
        {'params': list(model.bert.parameters()), 'lr': bert_lr},
        {'params': [p for n,p in model.named_parameters() if not n.startswith('bert')], 'lr': head_lr},
    ], weight_decay=0.01)

    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
    scaler = GradScaler('cuda', enabled=True)

    best_f1, best_epoch, patience_count = 0.0, 0, 0
    os.makedirs(f'outputs/ancient/{model_name}', exist_ok=True)
    best_ckpt = f'outputs/ancient/{model_name}/best.pt'

    for epoch in range(1, epochs+1):
        t0 = time.time()
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            token_type_ids = batch.get('token_type_ids')
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                loss, _ = model(input_ids, attention_mask, token_type_ids, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        dev_f1, _, _ = evaluate(model, dev_loader, use_crf)
        elapsed = time.time() - t0
        print(f'[{model_name}] Epoch {epoch}/{epochs} | loss={total_loss/len(train_loader):.4f} | dev_f1={dev_f1:.4f} | {elapsed:.0f}s', flush=True)

        if dev_f1 > best_f1 + 0.001:
            best_f1, best_epoch, patience_count = dev_f1, epoch, 0
            torch.save(model.state_dict(), best_ckpt)
            print(f'  ✅ New best! F1={best_f1:.4f}', flush=True)
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'  🛑 Early stopping at epoch {epoch}', flush=True)
                break
        torch.cuda.empty_cache()

    print(f'\n[{model_name}] Best: epoch={best_epoch} | dev_f1={best_f1:.4f}', flush=True)
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
    return model, best_ckpt

print('✅ Train functions defined', flush=True)

✅ Train functions defined


## Cell 6 — Train Model A: GuwenBERT + CRF

In [6]:
model_crf = GuwenBertCRF(BACKBONE, num_labels)
model_crf, ckpt_crf = train_model(model_crf, 'guwenbert_crf', epochs=20)

test_f1_crf, preds_crf, golds_crf = evaluate(model_crf, test_loader, use_crf=True)
print(f'\n[GuwenBERT+CRF] Test F1: {test_f1_crf:.4f}')
print(classification_report(golds_crf, preds_crf), flush=True)

pytorch_model.bin:   0%|          | 0.00/418M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ethanyt/guwenbert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/418M [00:00<?, ?B/s]

/tmp/ipykernel_24/219512635.py:71: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[guwenbert_crf] Epoch 1/20 | loss=33.0514 | dev_f1=0.6771 | 180s
  ✅ New best! F1=0.6771
[guwenbert_crf] Epoch 2/20 | loss=6.8032 | dev_f1=0.7309 | 179s
  ✅ New best! F1=0.7309
[guwenbert_crf] Epoch 3/20 | loss=4.6057 | dev_f1=0.7417 | 179s
  ✅ New best! F1=0.7417
[guwenbert_crf] Epoch 4/20 | loss=3.7040 | dev_f1=0.7407 | 178s
[guwenbert_crf] Epoch 5/20 | loss=3.0722 | dev_f1=0.7549 | 178s
  ✅ New best! F1=0.7549
[guwenbert_crf] Epoch 6/20 | loss=2.5970 | dev_f1=0.7540 | 178s
[guwenbert_crf] Epoch 7/20 | loss=2.2927 | dev_f1=0.7398 | 178s
[guwenbert_crf] Epoch 8/20 | loss=2.0128 | dev_f1=0.7587 | 178s
  ✅ New best! F1=0.7587
[guwenbert_crf] Epoch 9/20 | loss=1.8493 | dev_f1=0.7520 | 179s
[guwenbert_crf] Epoch 10/20 | loss=1.6893 | dev_f1=0.7549 | 178s
[guwenbert_crf] Epoch 11/20 | loss=1.5794 | dev_f1=0.7543 | 178s
[guwenbert_crf] Epoch 12/20 | loss=1.4474 | dev_f1=0.7525 | 178s
[guwenbert_crf] Epoch 13/20 | loss=1.3575 | dev_f1=0.7624 | 179s
  ✅ New best! F1=0.7624
[guwenbert_crf] Epo

## Cell 7 — Train Model B: GuwenBERT + Linear

In [7]:
model_linear = GuwenBertLinear(BACKBONE, num_labels)
model_linear, ckpt_linear = train_model(model_linear, 'guwenbert_linear', epochs=20)

test_f1_linear, preds_linear, golds_linear = evaluate(model_linear, test_loader, use_crf=False)
print(f'\n[GuwenBERT+Linear] Test F1: {test_f1_linear:.4f}')
print(classification_report(golds_linear, preds_linear), flush=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ethanyt/guwenbert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_24/219512635.py:71: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-

[guwenbert_linear] Epoch 1/20 | loss=0.7588 | dev_f1=0.6563 | 101s
  ✅ New best! F1=0.6563
[guwenbert_linear] Epoch 2/20 | loss=0.1876 | dev_f1=0.7133 | 101s
  ✅ New best! F1=0.7133
[guwenbert_linear] Epoch 3/20 | loss=0.1442 | dev_f1=0.7153 | 101s
  ✅ New best! F1=0.7153
[guwenbert_linear] Epoch 4/20 | loss=0.1193 | dev_f1=0.7258 | 102s
  ✅ New best! F1=0.7258
[guwenbert_linear] Epoch 5/20 | loss=0.1066 | dev_f1=0.7288 | 102s
  ✅ New best! F1=0.7288
[guwenbert_linear] Epoch 6/20 | loss=0.0958 | dev_f1=0.7267 | 101s
[guwenbert_linear] Epoch 7/20 | loss=0.0862 | dev_f1=0.7310 | 102s
  ✅ New best! F1=0.7310
[guwenbert_linear] Epoch 8/20 | loss=0.0781 | dev_f1=0.7403 | 101s
  ✅ New best! F1=0.7403
[guwenbert_linear] Epoch 9/20 | loss=0.0720 | dev_f1=0.7307 | 102s
[guwenbert_linear] Epoch 10/20 | loss=0.0665 | dev_f1=0.7374 | 102s
[guwenbert_linear] Epoch 11/20 | loss=0.0620 | dev_f1=0.7318 | 102s
[guwenbert_linear] Epoch 12/20 | loss=0.0570 | dev_f1=0.7388 | 101s
[guwenbert_linear] Epoch 

## Cell 8 — Error Analysis (Both Models)

In [8]:
from collections import defaultdict
import json

def error_analysis(preds, golds, model_name):
    print(f'\n{"="*60}')
    print(f'ERROR ANALYSIS — {model_name}')
    print('='*60)

    # 1. Per-entity F1
    print('\n[1] Per-entity F1:')
    print(classification_report(golds, preds))

    # 2. Token-level confusion matrix
    TYPES = ['DTM', 'LOC', 'ORG', 'PER', 'TITLE', 'O']
    confusion = defaultdict(lambda: defaultdict(int))
    for gold_seq, pred_seq in zip(golds, preds):
        for g, p in zip(gold_seq, pred_seq):
            g_type = g[2:] if g != 'O' else 'O'
            p_type = p[2:] if p != 'O' else 'O'
            if g_type != 'O' or p_type != 'O':
                confusion[g_type][p_type] += 1

    print('\n[2] Token Confusion Matrix (Gold→Pred):')
    header = f'{"":>8}' + ''.join(f'{t:>8}' for t in TYPES)
    print(header)
    for g in TYPES:
        if g == 'O': continue
        row = f'{g:>8}' + ''.join(f'{confusion[g][p]:>8}' for p in TYPES)
        print(row)

    # 3. FP/FN stats
    print('\n[3] FP / FN per entity type:')
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for gold_seq, pred_seq in zip(golds, preds):
        for g, p in zip(gold_seq, pred_seq):
            g_type = g[2:] if g != 'O' else None
            p_type = p[2:] if p != 'O' else None
            if g_type and g_type == p_type: tp[g_type] += 1
            if p_type and p_type != g_type: fp[p_type] += 1
            if g_type and g_type != p_type: fn[g_type] += 1

    print(f'{"Type":>8} {"Support":>9} {"FP":>8} {"FN":>8} {"FP%":>8} {"FN%":>8}')
    for etype in sorted(tp.keys()):
        support = tp[etype] + fn[etype]
        fp_pct  = fp[etype] / (tp[etype]+fp[etype]+1e-9) * 100
        fn_pct  = fn[etype] / (support+1e-9) * 100
        print(f'{etype:>8} {support:>9,} {fp[etype]:>8,} {fn[etype]:>8,} {fp_pct:>7.1f}% {fn_pct:>7.1f}%')

    # 4. Common error spans
    print('\n[4] Most common FP spans (over-predicted):')
    fp_spans = defaultdict(lambda: defaultdict(int))
    for gold_seq, pred_seq in zip(golds, preds):
        cur, cur_type = [], None
        cur_g = []
        for g, p in zip(gold_seq, pred_seq):
            pass  # simplified — just count entity-level errors
    
    # Save results
    results = {
        'model': model_name,
        'test_f1': test_f1_crf if 'CRF' in model_name else test_f1_linear,
        'fp': dict(fp), 'fn': dict(fn), 'tp': dict(tp)
    }
    with open(f'outputs/ancient/{model_name.lower().replace(" ","_")}_error.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n✅ Saved error analysis', flush=True)

error_analysis(preds_crf,    golds_crf,    'GuwenBERT + CRF')
error_analysis(preds_linear, golds_linear, 'GuwenBERT + Linear')


ERROR ANALYSIS — GuwenBERT + CRF

[1] Per-entity F1:
              precision    recall  f1-score   support

         DTM       0.79      0.88      0.83       176
         LOC       0.90      0.94      0.92      5029
         ORG       0.91      0.96      0.94      2047
         PER       0.94      0.96      0.95     13865
       TITLE       0.83      0.91      0.87      2812

   micro avg       0.91      0.95      0.93     23929
   macro avg       0.87      0.93      0.90     23929
weighted avg       0.91      0.95      0.93     23929


[2] Token Confusion Matrix (Gold→Pred):
             DTM     LOC     ORG     PER   TITLE       O
     DTM     376      14       0       5       0      15
     LOC      19    8996      26      59      43     191
     ORG       0      25    2474      39       6      14
     PER       2     119      45   24702     197     342
   TITLE       0      85       2     113    9041     197

[3] FP / FN per entity type:
    Type   Support       FP       FN      FP

## Cell 9 — Final Comparison

In [9]:
print('='*60)
print('FINAL RESULTS — Ancient Chinese NER')
print('='*60)
print(f'Dataset: AnChineseNERE + CHisIEC + BEEVETA + C-CLUE')
print(f'  Train: 11,386 | Dev: 1,697 | Test: 3,076 sentences')
print(f'  Entity types: PER / LOC / TITLE / DTM / ORG')
print(f'\nModel               Test F1   Delta')
print(f'─────────────────── ─────── ───────')
print(f'GuwenBERT + Linear  {test_f1_linear:.4f}')
print(f'GuwenBERT + CRF     {test_f1_crf:.4f}   {test_f1_crf-test_f1_linear:+.4f}')
print(f'\nCheckpoints saved:')
print(f'  CRF:    {ckpt_crf}')
print(f'  Linear: {ckpt_linear}', flush=True)

FINAL RESULTS — Ancient Chinese NER
Dataset: AnChineseNERE + CHisIEC + BEEVETA + C-CLUE
  Train: 11,386 | Dev: 1,697 | Test: 3,076 sentences
  Entity types: PER / LOC / TITLE / DTM / ORG

Model               Test F1   Delta
─────────────────── ─────── ───────
GuwenBERT + Linear  0.9002
GuwenBERT + CRF     0.9321   +0.0319

Checkpoints saved:
  CRF:    outputs/ancient/guwenbert_crf/best.pt
  Linear: outputs/ancient/guwenbert_linear/best.pt


## Cell 10 — Save & Download

In [10]:
import os
os.chdir('/kaggle/working/chisiec-ner')

!zip -r /kaggle/working/ancient_ner_models.zip \
    outputs/ancient/ \
    data/ancient/

!ls -lh /kaggle/working/ancient_ner_models.zip
print('✅ Ready to download', flush=True)

  adding: outputs/ancient/ (stored 0%)
  adding: outputs/ancient/guwenbert_crf/ (stored 0%)
  adding: outputs/ancient/guwenbert_crf/best.pt (deflated 12%)
  adding: outputs/ancient/guwenbert_+_crf_error.json (deflated 50%)
  adding: outputs/ancient/guwenbert_+_linear_error.json (deflated 49%)
  adding: outputs/ancient/guwenbert_linear/ (stored 0%)
  adding: outputs/ancient/guwenbert_linear/best.pt (deflated 12%)
  adding: data/ancient/ (stored 0%)
  adding: data/ancient/dev.txt (deflated 71%)
  adding: data/ancient/test.txt (deflated 73%)
  adding: data/ancient/train.txt (deflated 72%)
-rw-r--r-- 1 root root 703M Apr  5 16:49 /kaggle/working/ancient_ner_models.zip
✅ Ready to download
